In [10]:
#Natural languge processing

#Es necesario transcribir el audio, para ello, se utiliza la libreria whsper

!pip install openai-whisper
!pip install python-docx
!pip install transformers
!python -m spacy download es_core_news_sm
from google.colab import files

import whisper
import spacy
import docx
from transformers import pipeline #analisis de sentimientos

modelo=whisper.load_model("base") #el modelo base es el mas rapido
nlp=spacy.load("es_core_news_sm")

# Initialize Spanish sentiment analysis pipeline
sentiment_pipeline = pipeline('sentiment-analysis', model='finiteautomata/beto-sentiment-analysis')

print("Whisper model, Spacy NLP model, and Sentiment Analysis model loaded successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 96.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Whisper model, Spacy NLP model, and Sentiment Analysis model loaded successfully.


In [11]:
# This cell is no longer needed as python-docx installation is moved to YBQL4E18ewQq
# !pip install python-docx

First, let's upload an audio file. You can use the file uploader below.

In [12]:
from google.colab import files

uploaded = files.upload()

audio_file_paths = []
for fn in uploaded.keys():
  audio_file_paths.append(fn)
  print(f'User uploaded file: "{fn}"')

if not audio_file_paths:
    print("No audio files were uploaded.")
else:
    print(f"Uploaded {len(audio_file_paths)} audio file(s).")

Saving Colaborador 1.m4a to Colaborador 1 (9).m4a
Saving Colaborador 2.m4a to Colaborador 2 (8).m4a
Saving Colaborador 3.m4a to Colaborador 3 (7).m4a
Saving Colaborador 4.m4a to Colaborador 4 (7).m4a
Saving Colaborador 5.m4a to Colaborador 5 (7).m4a
Saving Colaborador 6.m4a to Colaborador 6 (7).m4a
User uploaded file: "Colaborador 1 (9).m4a"
User uploaded file: "Colaborador 2 (8).m4a"
User uploaded file: "Colaborador 3 (7).m4a"
User uploaded file: "Colaborador 4 (7).m4a"
User uploaded file: "Colaborador 5 (7).m4a"
User uploaded file: "Colaborador 6 (7).m4a"
Uploaded 6 audio file(s).


Now that we have the audio file, let's use the `whisper` model to transcribe it.

In [14]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
from docx.shared import Inches

if 'audio_file_paths' in locals() and audio_file_paths:
    generated_doc_filenames = [] # List to store filenames of generated documents
    for audio_path in audio_file_paths:
        print(f"\nTranscribing audio from '{audio_path}', this may take a few minutes...")
        resultado = modelo.transcribe(audio_path)
        transcribed_text = resultado['text']
        print(f"Transcription complete for '{audio_path}'.")

        # Process the text with SpaCy
        doc = nlp(transcribed_text)

        # Extract key concepts (lemmas of nouns, verbs, adjectives)
        palabras_importantes = []
        for token in doc:
            if not token.is_stop and not token.is_punct and not token.is_space and token.pos_ in ['NOUN', 'VERB', 'ADJ']:
                palabras_importantes.append(token.lemma_)

        # Get top 50 unique key concepts
        if palabras_importantes:
            from collections import Counter
            word_counts = Counter(palabras_importantes)
            top_50_palabras = [word for word, count in word_counts.most_common(50)]
            texto_claves = ', '.join(top_50_palabras)
        else:
            texto_claves = "No se encontraron palabras clave."

        # Extract themes (noun chunks)
        noun_chunks = [chunk.text for chunk in doc.noun_chunks]

        # Get top 20 unique noun chunks
        if noun_chunks:
            from collections import Counter
            chunk_counts = Counter(noun_chunks)
            top_20_chunks = [chunk for chunk, count in chunk_counts.most_common(20)]
            top_noun_chunks = ', '.join(top_20_chunks)
        else:
            top_noun_chunks = "No se encontraron temas."

        # --- Sentiment Analysis ---#
        # Analyze sentiment for each sentence
        sentences = [sent.text for sent in doc.sents]
        sentence_sentiments = []
        if sentences:
            for sent in sentences:
                # Add truncation=True to handle long sequences
                sent_result = sentiment_pipeline(sent, truncation=True)
                sentence_sentiments.append(sent_result[0]['label'])

            # Convert to DataFrame for easier analysis
            sentiment_df = pd.DataFrame(sentence_sentiments, columns=['Sentiment'])

            # Calculate the proportion of each sentiment
            sentiment_counts = sentiment_df['Sentiment'].value_counts(normalize=True) * 100
            sentiment_counts = sentiment_counts.reindex(['POS', 'NEU', 'NEG'], fill_value=0) # Ensure all categories are present

            # Determine the most common sentiment
            most_common_sentiment = sentiment_counts.idxmax()
            sentiment_summary_text = f"El sentimiento predominante es '{most_common_sentiment}' con {sentiment_counts.max():.2f}% de las oraciones.\nDistribución de Sentimientos:\n{sentiment_counts.to_string(float_format='%.2f')}"

            # Plotting sentiment distribution
            plt.figure(figsize=(8, 5))
            sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, hue=sentiment_counts.index, palette='viridis', legend=False)
            plt.title('Distribución de Sentimientos en el Texto')
            plt.xlabel('Sentimiento')
            plt.ylabel('Porcentaje de Oraciones')
            plt.ylim(0, 100)
            plt.grid(axis='y', linestyle='--', alpha=0.7)

            # Save plot to a BytesIO object
            img_buffer = io.BytesIO()
            plt.savefig(img_buffer, format='png', bbox_inches='tight')
            img_buffer.seek(0)
            plt.close() # Close the plot to free memory
        else:
            sentiment_summary_text = "No se pudo realizar el análisis de sentimientos porque no se detectaron oraciones."
            img_buffer = None

        # --- Generate Word Document ---#
        document = docx.Document()
        document.add_heading(f'Análisis de Entrevista: {os.path.basename(audio_path)}', level=1)

        document.add_heading('Texto Transcrito:', level=2)
        document.add_paragraph(transcribed_text)

        document.add_heading('Conceptos Clave (Lemmas):', level=2)
        document.add_paragraph(texto_claves)

        document.add_heading('Temas Principales (Noun Chunks):', level=2)
        document.add_paragraph(top_noun_chunks)

        document.add_heading('Análisis de Sentimientos:', level=2)
        document.add_paragraph(sentiment_summary_text)
        if img_buffer:
            document.add_picture(img_buffer, width=Inches(6))

        # Sanitize filename for output
        base_name = os.path.basename(audio_path)
        sanitized_base_name = base_name.replace(' ', '_').replace('.', '_')
        output_filename = f"Analisis_Entrevista_{sanitized_base_name}.docx"
        document.save(output_filename)
        print(f"Documento '{output_filename}' generado con éxito.")
        generated_doc_filenames.append(output_filename)

    print("\nTodos los documentos han sido procesados.")
    # The following print statement will be replaced by a new cell for ZIP download
    # print(f"Documentos generados: {generated_doc_filenames}")

else:
    print("No se cargaron archivos de audio para transcribir y analizar.")


Transcribing audio from 'Colaborador 1 (9).m4a', this may take a few minutes...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription complete for 'Colaborador 1 (9).m4a'.
Documento 'Analisis_Entrevista_Colaborador_1_(9)_m4a.docx' generado con éxito.

Transcribing audio from 'Colaborador 2 (8).m4a', this may take a few minutes...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription complete for 'Colaborador 2 (8).m4a'.
Documento 'Analisis_Entrevista_Colaborador_2_(8)_m4a.docx' generado con éxito.

Transcribing audio from 'Colaborador 3 (7).m4a', this may take a few minutes...


...

Transcription complete for 'Colaborador 4 (7).m4a'.
Documento 'Analisis_Entrevista_Colaborador_4_(7)_m4a.docx' generado con éxito.

Transcribing audio from 'Colaborador 5 (7).m4a', this may take a few minutes...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription complete for 'Colaborador 5 (7).m4a'.
Documento 'Analisis_Entrevista_Colaborador_5_(7)_m4a.docx' generado con éxito.

Transcribing audio from 'Colaborador 6 (7).m4a', this may take a few minutes...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription complete for 'Colaborador 6 (7).m4a'.
Documento 'Analisis_Entrevista_Colaborador_6_(7)_m4a.docx' generado con éxito.

Todos los documentos han sido procesados.


In [15]:
import zipfile

if 'generated_doc_filenames' in locals() and generated_doc_filenames:
    zip_filename = 'Analisis_Entrevistas.zip'
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        for filename in generated_doc_filenames:
            zipf.write(filename, os.path.basename(filename))

    print(f"'{zip_filename}' creado con éxito. Descargando...")
    files.download(zip_filename)
else:
    print("No se encontraron documentos para comprimir y descargar.")

'Analisis_Entrevistas.zip' creado con éxito. Descargando...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Below is the modified cell to include the Word document generation for each transcribed audio file.